Generate the numerical data for the three publication figures in the local `results` folder. Existing results are checked and not overwritten. The extended calculations can take a long time; each death rate is saved separately when complete.


In [1]:
import numpy as np
from tqdm import tqdm
from config import results_loc
from model import get_correct_M, get_invasion_probability, get_qsd, get_stationary_distribution

results_loc.mkdir(parents=True, exist_ok=True)


In [2]:
def generate_fixation_data(filename, rs, deathrate, K=100):
    output = results_loc / filename
    if output.exists():
        with np.load(output) as data:
            if (not np.array_equal(data["rs"], rs)
                    or data["probabilities"].shape != rs.shape
                    or not np.all(np.isfinite(data["probabilities"]))
                    or data["deathrate"] != deathrate or data["K"] != K):
                raise ValueError(f"Incompatible existing result: {output}")
        print(f"Already exists: {output}")
        return
    M = get_correct_M(1, deathrate, K)
    stationary = get_stationary_distribution(1, deathrate, M)
    weights = np.arange(len(stationary)) * stationary
    probabilities = []
    for r in tqdm(rs, desc=f"d={deathrate}"):
        rho = get_invasion_probability(r, deathrate, M)
        probability = np.dot(rho[:len(weights)], weights) / weights.sum()
        if not np.isfinite(probability):
            raise ValueError(f"Non-finite probability for r={r}, d={deathrate}")
        probabilities.append(probability)
    with output.open("xb") as file:
        np.savez(file, rs=rs, probabilities=probabilities, deathrate=deathrate, K=K, M=M)
    print(f"Generated: {output}")


In [3]:
for M in [25, 50, 100, 200]:
    output = results_loc / f"qsd_M{M}_d0.5.npz"
    if output.exists():
        with np.load(output) as data:
            if (data["distribution"].shape != (M,) or data["M"] != M
                    or data["deathrate"] != 0.5
                    or not np.all(np.isfinite(data["distribution"]))):
                raise ValueError(f"Incompatible existing result: {output}")
        print(f"Already exists: {output}")
        continue
    distribution = get_qsd(1, 0.5, M)
    with output.open("xb") as file:
        np.savez(file, distribution=distribution, M=M, deathrate=0.5)
    print(f"Generated: {output}")


Already exists: /Users/george/Documents/EvoLudoWrapperProject/Publishing/NonEvoludo/DifferenceEqs/constantFitness/results/qsd_M25_d0.5.npz
Already exists: /Users/george/Documents/EvoLudoWrapperProject/Publishing/NonEvoludo/DifferenceEqs/constantFitness/results/qsd_M50_d0.5.npz
Already exists: /Users/george/Documents/EvoLudoWrapperProject/Publishing/NonEvoludo/DifferenceEqs/constantFitness/results/qsd_M100_d0.5.npz
Already exists: /Users/george/Documents/EvoLudoWrapperProject/Publishing/NonEvoludo/DifferenceEqs/constantFitness/results/qsd_M200_d0.5.npz


In [4]:
standard_rs = np.array([0.8, 0.85, 0.9, 0.95, 0.99, 1.0, 1.01, 1.05, 1.1, 1.15,
                        1.2, 1.25, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0])
generate_fixation_data("well_mixed_standard_d0.3.npz", standard_rs, 0.3)


Already exists: /Users/george/Documents/EvoLudoWrapperProject/Publishing/NonEvoludo/DifferenceEqs/constantFitness/results/well_mixed_standard_d0.3.npz


In [5]:
low_rs = np.array([0.98, 0.985, 0.99, 0.995, 0.999, 1.001, 1.01, 1.025, 1.05, 1.075, 1.1, 1.125, 1.15])
high_rs = np.array([1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0, 3.5, 4.0, 4.5,
                    5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 15, 20, 25, 30, 35, 40, 45, 50,
                    75, 100, 200, 400, 800, 1600, 3200, 6400])
for d in [0.01, 0.1, 0.3, 0.5]:
    generate_fixation_data(f"well_mixed_extended_d{d}.npz", np.concatenate((low_rs, high_rs)), d)


d=0.01: 100%|██████████| 48/48 [01:48<00:00,  2.25s/it]


Generated: /Users/george/Documents/EvoLudoWrapperProject/Publishing/NonEvoludo/DifferenceEqs/constantFitness/results/well_mixed_extended_d0.01.npz


d=0.1: 100%|██████████| 48/48 [02:23<00:00,  2.99s/it]


Generated: /Users/george/Documents/EvoLudoWrapperProject/Publishing/NonEvoludo/DifferenceEqs/constantFitness/results/well_mixed_extended_d0.1.npz


d=0.3: 100%|██████████| 48/48 [05:58<00:00,  7.48s/it]


Generated: /Users/george/Documents/EvoLudoWrapperProject/Publishing/NonEvoludo/DifferenceEqs/constantFitness/results/well_mixed_extended_d0.3.npz


d=0.5: 100%|██████████| 48/48 [21:42<00:00, 27.14s/it]

Generated: /Users/george/Documents/EvoLudoWrapperProject/Publishing/NonEvoludo/DifferenceEqs/constantFitness/results/well_mixed_extended_d0.5.npz
